# Loan Approval Prediction Assignment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score

df=pd.read_excel('loan_approval.xlsx')
df.head()

In [ ]:
# Q1-Q10 Solution
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
for c in df.columns:
    if df[c].dtype=='object':
        df[c]=df[c].fillna(df[c].mode()[0])
    else:
        df[c]=df[c].fillna(df[c].median())
cat=df.select_dtypes(include='object').columns.tolist()
print(cat)
target=df.columns[-1]
sns.countplot(x=target,data=df); plt.show()
num=df.select_dtypes(include='number').columns
if len(num)>0:
    sns.boxplot(y=df[num[0]]); plt.show()
    sns.boxplot(x=target,y=num[0],data=df); plt.show()
if 'Years of Employment' in df.columns:
    sns.boxplot(x=target,y='Years of Employment',data=df); plt.show()
for c in num:
    q1,q3=df[c].quantile([0.25,0.75]);iqr=q3-q1
    l,u=q1-1.5*iqr,q3+1.5*iqr
    df[c]=df[c].clip(l,u)
if df[target].dtype=='object':
    df[target]=LabelEncoder().fit_transform(df[target])
for c in df.select_dtypes(include='object').columns:
    if c!=target:
        df[c]=LabelEncoder().fit_transform(df[c].astype(str))
X=df.drop(columns=[target]); y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
sc=StandardScaler()
X_train=sc.fit_transform(X_train); X_test=sc.transform(X_test)
model=LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)
pred=model.predict(X_test)
proba=model.predict_proba(X_test)[:,1]
cm=confusion_matrix(y_test,pred)
ConfusionMatrixDisplay(cm).plot(); plt.show()
fpr,tpr,_=roc_curve(y_test,proba)
plt.plot(fpr,tpr); plt.plot([0,1],[0,1],'--'); plt.show()
print('AUC:',roc_auc_score(y_test,proba))
print('Intercept:',model.intercept_)
print('Coefficients:',dict(zip(X.columns,model.coef_[0])))